# Rich-circuit scaling sweep

LR sweep, then main scaling runs on a rich circuit (all wires tapped at uniform depths, all outputs trained, online data). Same recipe as `colab_sweep.ipynb`. Runs are idempotent: rerun any cell to resume.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

import jax
print(jax.devices())

In [ ]:
import json
from pathlib import Path

import numpy as np

from train import RunConfig, run

N_WIRES, CIRC_DEPTH, CIRCUIT_SEED = 100, 8, 0
GRID = [(32, 2), (48, 2), (64, 3), (96, 3), (128, 4),
        (180, 5), (256, 6), (360, 7), (512, 8)]
LR_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
TUNE_STEPS, FULL_STEPS, BATCH = 5_000, 50_000, 256
OUT_DIR = f"runs/rich_c{N_WIRES}x{CIRC_DEPTH}"
LR_TABLE = Path(OUT_DIR) / "lr_table.json"


def cfg(width, depth, lr, steps, out_dir, seed=0):
    return RunConfig(width=width, mlp_depth=depth, lr=lr, steps=steps, batch=BATCH,
                     n_wires=N_WIRES, circ_depth=CIRC_DEPTH,
                     circuit_seed=CIRCUIT_SEED, model_seed=seed, out_dir=out_dir)

## LR sweep

In [ ]:
table = {}
for w, d in GRID:
    losses = {}
    for lr in LR_GRID:
        c = cfg(w, d, lr, TUNE_STEPS, f"{OUT_DIR}/tune")
        run(c)
        losses[lr] = float(np.load(c.npz_path)["per_out_loss"][-1].mean())
    table[f"w{w}d{d}"] = best = min(losses, key=losses.get)
    edge = "  <- grid edge" if best in (LR_GRID[0], LR_GRID[-1]) else ""
    print(f"w{w}d{d}: best lr {best:g}{edge}")
LR_TABLE.write_text(json.dumps(table, indent=2))

## Main runs

In [ ]:
table = json.loads(LR_TABLE.read_text())
for w, d in GRID:
    run(cfg(w, d, table[f"w{w}d{d}"], FULL_STEPS, OUT_DIR))